<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/937_IRMv3_NodeTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Node Tests

This test file verifies the **execution layer of the orchestrator**.

Where the utilities tests validated:

* scoring
* rollups
* report generation

these node tests validate:

* node behavior
* closure construction
* state passing
* filesystem side effects

This is exactly the right level of testing for orchestration logic.

---

# What This Test Suite Does Well

## 1. Goal Node Validation

```python
test_goal_node_sets_goal()
```

This confirms the orchestrator initializes its mission correctly.

You're testing:

* presence of goal
* correct structure
* meaningful name content

That protects the **agent contract**.

If someone accidentally changes the node output format later, the test suite catches it.

---

# 2. Planning Node Validation

```python
test_planning_node_sets_plan()
```

This verifies:

* plan exists
* correct data type
* minimum steps

The plan is essentially **documentation encoded in the agent**, so testing it is valuable.

Many teams skip this level of validation.

---

# 3. Data Loading Node (Closure Test)

Your data loader node is a **closure factory**, so testing it is important.

You correctly test:

### Empty directory case

```python
test_data_loading_node_empty_dir()
```

Ensures the orchestrator remains resilient when telemetry is missing.

This protects production behavior.

---

### Minimal dataset case

```python
test_data_loading_node_loads_minimal_data()
```

This verifies the closure correctly:

* reads JSON
* builds lookup tables
* injects results into state

That confirms the **data ingestion node works independently of the graph**.

---

# 4. Scoring Node Tests

This is a particularly strong section.

You test the node using a **mock state**, which mirrors real orchestrator input.

Example:

```python
state: IRMv3State = {
    "integration_health_logs": [],
    ...
}
```

This ensures the node:

* accepts state correctly
* returns the expected keys
* produces a valid portfolio status

You're validating the **state transformation contract**.

That’s exactly what node tests should do.

---

# Governance Failure Scenario

This test is excellent:

```python
test_scoring_node_critical_when_governance_low()
```

You simulate:

* high uptime
* many open risks
* overdue remediations

Then verify governance score degrades.

This confirms the orchestrator can detect **organizational breakdowns**.

That’s a core design goal of IRMv3.

---

# 5. Report Node Tests

This is another strong section.

Your report node is a **closure with filesystem side effects**, which can be tricky to test.

You handle this well using `tmp_path`.

Example:

```python
config.reports_dir = str(tmp_path)
```

This isolates the test environment.

The test verifies:

* report file created
* correct output path returned
* report content generated

That covers the most important behaviors.

---

# File Creation Verification

You also verify that the file actually exists:

```python
path.exists()
```

And that the content includes key signals:

```python
assert "Portfolio Health" in content
```

This protects the **report pipeline contract**.

---

# Architectural Strength

What I like most about this test file is that it verifies **node behavior without needing the graph**.

Example pattern:

```python
node = make_scoring_node(config)
out = node(state)
```

This isolates each node and tests it as a **pure transformation function**.

This is the correct way to test LangGraph nodes.

---

# How This Complements the Utilities Tests

Your testing architecture now looks like this:

```
Utilities Tests
    ↓
Test scoring logic
Test rollups
Test report generation
Test data loader

Node Tests
    ↓
Test node closures
Test state transformations
Test filesystem outputs
```

Then your **third test suite** will likely validate the full graph.

This layered testing approach is excellent.

---

# One Minor Improvement I Would Suggest

You could add a test verifying **state override behavior**.

Example:

```python
def test_data_loading_node_respects_state_data_dir(tmp_path):
    config = IRMv3Config()
    config.data_dir = "wrong_dir"
    node = make_data_loading_node(config, str(tmp_path))
    state = {"data_dir": str(tmp_path)}
    out = node(state)
    assert out["loader_record_counts"]["agents"] == 0
```

This verifies that **state overrides config**, which your node design supports.

---

# Another Nice Optional Test

You might add a test ensuring **report filenames are unique**.

Example:

```python
def test_report_node_generates_timestamped_file(tmp_path):
```

But this is not essential.

---

# Portfolio Perspective

From a **GitHub portfolio perspective**, this file shows:

* you understand orchestration architecture
* you test closure-based nodes
* you isolate filesystem side effects
* you verify state transformations

That’s exactly how production orchestration systems are validated.

Very few AI repos include this level of testing.

---

# Final Verdict

This is a **high-quality node test suite**.

Strengths:

✓ Closure node testing
✓ State transformation validation
✓ Filesystem side-effect testing
✓ Governance failure simulation
✓ Clean isolation via `tmp_path`

Together with your utilities tests, the orchestrator pipeline is now **very well protected**.



In [ ]:
"""
Node tests for IRMv3: goal, planning, data_loading (closure), scoring (closure), report (closure).
Run from project root: python -m pytest test_irm_v3_nodes.py -v --tb=short
"""
import json
from pathlib import Path

import pytest

_root = Path(__file__).resolve().parent
if str(_root) not in __import__("sys").path:
    __import__("sys").path.insert(0, str(_root))

from config import IRMv3Config, IRMv3State
from agents.irm_v3.orchestrator.nodes import (
    goal_node,
    planning_node,
    make_data_loading_node,
    make_scoring_node,
    make_report_node,
)


# ----- Goal and planning -----


def test_goal_node_sets_goal():
    state: IRMv3State = {}
    out = goal_node(state)
    assert "goal" in out
    assert "name" in out["goal"]
    assert "Integration" in out["goal"].get("name", "") or "Risk" in out["goal"].get("name", "")


def test_planning_node_sets_plan():
    state: IRMv3State = {}
    out = planning_node(state)
    assert "plan" in out
    assert isinstance(out["plan"], list)
    assert len(out["plan"]) >= 3


# ----- Data loading node (closure) -----


def test_data_loading_node_empty_dir(tmp_path):
    config = IRMv3Config()
    config.data_dir = str(tmp_path)
    node = make_data_loading_node(config, str(tmp_path))
    state: IRMv3State = {}
    out = node(state)
    assert out["agents"] == []
    assert "agents_lookup" in out
    assert "loader_record_counts" in out
    assert out["loader_record_counts"].get("agents") == 0
    assert "validation_warnings" in out


def test_data_loading_node_loads_minimal_data(tmp_path):
    (tmp_path / "agents.json").write_text(json.dumps([
        {"agent_id": "a1", "agent_name": "A1"},
    ]))
    (tmp_path / "workflows.json").write_text(json.dumps([]))
    config = IRMv3Config()
    config.data_dir = str(tmp_path)
    node = make_data_loading_node(config, str(tmp_path))
    state: IRMv3State = {}
    out = node(state)
    assert len(out["agents"]) == 1
    assert out["agents_lookup"]["a1"]["agent_name"] == "A1"


# ----- Scoring node (closure) -----


def test_scoring_node_with_mock_state():
    config = IRMv3Config()
    node = make_scoring_node(config)
    state: IRMv3State = {
        "integration_health_logs": [],
        "dependency_graph": [],
        "dependency_incidents": [],
        "workflows_by_agent": {},
        "system_integrations": [],
        "remediation_actions": [
            {"status": "resolved", "days_open": 1, "resolution_date": "2026-01-01", "missed_deadline_flag": False},
        ],
        "risk_signals": [],
        "task_execution_logs": [],
        "workflows": [],
        "expected_vs_actual_value": None,
        "kpis_cost": None,
    }
    out = node(state)
    assert "integration_stability_score" in out
    assert "portfolio_status" in out
    assert out["portfolio_status"] in ("HEALTHY", "AT_RISK", "CRITICAL")
    assert "executive_triggers" in out
    assert "score_rollup" in out


def test_scoring_node_critical_when_governance_low():
    config = IRMv3Config()
    node = make_scoring_node(config)
    state: IRMv3State = {
        "integration_health_logs": [{"integration_id": "i1", "uptime_percent": 99, "latency_ms": 200, "error_rate_percent": 0, "schema_mismatch_events": 0}],
        "dependency_graph": [],
        "dependency_incidents": [],
        "workflows_by_agent": {},
        "system_integrations": [],
        "remediation_actions": [
            {"status": "open", "missed_deadline_flag": True},
            {"status": "open", "missed_deadline_flag": True},
        ],
        "risk_signals": [{"status": "open"}] * 10,
        "task_execution_logs": [],
        "workflows": [],
        "expected_vs_actual_value": None,
        "kpis_cost": None,
    }
    out = node(state)
    # Governance should be low; portfolio may be CRITICAL or AT_RISK
    assert out["governance_responsiveness_score"] < 80
    assert out["portfolio_status"] in ("CRITICAL", "AT_RISK")


# ----- Report node (closure) -----


def test_report_node_writes_file(tmp_path):
    config = IRMv3Config()
    config.reports_dir = str(tmp_path)
    node = make_report_node(config, str(_root))
    state: IRMv3State = {
        "portfolio_status": "HEALTHY",
        "portfolio_score": 80,
        "integration_stability_score": 85,
        "dependency_exposure_score": 82,
        "governance_responsiveness_score": 78,
        "automation_value_score": 80,
        "score_rollup": {},
        "executive_triggers": [],
        "loader_record_counts": {},
        "data_snapshot_loaded_at": None,
        "validation_warnings": [],
        "mission_runs": None,
        "risk_signals": None,
        "remediation_actions": None,
    }
    out = node(state)
    assert "report_file_path" in out
    assert "mission_report" in out
    path = Path(out["report_file_path"])
    assert path.exists()
    content = path.read_text()
    assert "Portfolio Health" in content
    assert "HEALTHY" in content
